# فصل اول — آزمایش عملی مهندسی داده
این دفترچه از داده‌های **مصنوعی** استفاده می‌کند. برنامه را در پوشهٔ `downloads` اجرا کنید. وابستگی خارجی ندارد.

## ۱. مشاهدهٔ دادهٔ خام
ابتدا چند سطر فایل را مشاهده می‌کنیم. تکرار، مبلغ نامعتبر و تاریخ غیریکسان عمداً در داده وجود دارند.

In [1]:
from pathlib import Path
import csv
from pprint import pprint

csv_path = Path("orders_sample.csv")
with csv_path.open(encoding="utf-8", newline="") as f:
    raw_rows = list(csv.DictReader(f))
print("Raw rows:", len(raw_rows))
pprint(raw_rows[:4])

Raw rows: 10
[{'amount_toman': '120000',
  'city': 'Tehran',
  'order_date': '2026-10-15',
  'order_id': '1001',
  'status': 'paid'},
 {'amount_toman': '80000',
  'city': 'Isfahan',
  'order_date': '2026-10-15',
  'order_id': '1002',
  'status': 'paid'},
 {'amount_toman': '50000',
  'city': 'Tehran',
  'order_date': '2026-10-15',
  'order_id': '1003',
  'status': 'canceled'},
 {'amount_toman': '',
  'city': 'Shiraz',
  'order_date': '2026-10-15',
  'order_id': '1004',
  'status': 'paid'}]


## ۲. پاک‌سازی و ثبت در SQLite
قاعده: کلید سفارش یکتا، تاریخ قابل تبدیل، مبلغ غیرمنفی و وضعیت شناخته‌شده؛ خطاها در پروندهٔ قرنطینه ثبت می‌شوند.

In [2]:
from chapter01_examples import run_pipeline

output_dir = Path("example_output_notebook")
result = run_pipeline(csv_path, output_dir)
pprint(result)

{'accepted_records': 7,
 'exact_duplicates': 1,
 'rejected_records': 2,
 'report': [{'city': 'Isfahan', 'paid_orders': 2, 'revenue_toman': 150000},
            {'city': 'Tehran', 'paid_orders': 3, 'revenue_toman': 190000}],
 'source_records': 10,
 'stored_records': 7,
 'total_paid_revenue_toman': 340000}


## ۳. پرس‌وجوی مستقیم SQL
فقط سفارش‌های با وضعیت `paid` در شاخص فروش پرداخت‌شده قرار می‌گیرند.

In [3]:
import sqlite3
with sqlite3.connect(output_dir / "orders.db") as conn:
    rows = conn.execute("SELECT city,COUNT(*),SUM(amount_toman) FROM orders WHERE status=? GROUP BY city ORDER BY city", ("paid",)).fetchall()
print(rows)

[('Isfahan', 2, 150000), ('Tehran', 3, 190000)]


## ۴. آزمون اعداد و اجرای مجدد
تکرار اجرای فایل نباید رکورد یا مبلغ فروش را افزایش دهد (Idempotency).

In [4]:
assert result["source_records"] == 10
assert result["stored_records"] == 7
assert result["exact_duplicates"] == 1
assert result["rejected_records"] == 2
assert result["total_paid_revenue_toman"] == 340000
assert rows == [("Isfahan", 2, 150000), ("Tehran", 3, 190000)]
second = run_pipeline(csv_path, output_dir)
assert second == result
print("ALL CHECKS PASSED; repeat run changed no totals")

ALL CHECKS PASSED; repeat run changed no totals


## ۵. معنای نتیجه و محدودیت
مجموع ۳۴۰ هزار تومان فقط حاصل **سفارش‌های ثبت‌شده با وضعیت paid** در دادهٔ ساختگی است؛ این عدد لزوماً درآمد خالص حسابداری نیست. برگشت وجه، مالیات، کارمزد، نرخ ارز و مبالغ اصلاحی در این نمونه مدل نشده‌اند. برای محیط تولیدی باید نحوهٔ تغییر وضعیت سفارش و رخدادهای برگشت وجه طراحی شود.